# project overivew
user will send file which have multiple lines of text and after then my python script will goes through each line and assign weight to each sentence and after that it responds with one important highlights and a summary.

In [76]:
from typing import Text
import os
import google.generativeai as genai
from google.colab import userdata
import json

In [77]:
class TextAnalyser:

  def __init__(self, file_path=None):
    self.file_path = file_path
    self.size = 0
    self.text = []
    # Retrieve API key securely from Colab secrets with a clear name
    self.__api_key = userdata.get('GOOGLE_API_KEY')

    genai.configure(api_key=self.__api_key)
    self.__model = genai.GenerativeModel('gemini-3-flash-preview')

    if self.file_path is not None: # Check if a file path was provided
      try:
        self.size = os.path.getsize(self.file_path)
        if self.size <= 10000:
          with open(self.file_path, 'r') as f:
            self.text = f.readlines() # Store text as instance variable
          print(f"Successfully loaded file '{self.file_path}' ({self.size} bytes).")
        else:
          print(f'File "{self.file_path}" (size: {self.size} bytes) is outside the allowed range (1KB-10KB).')
      except FileNotFoundError:
        print(f"Error: File '{self.file_path}' not found.")
      except Exception as e:
        print(f"An error occurred while processing file '{self.file_path}': {e}")
    else:
      print('No file path provided. TextAnalyser initialized without a file.')

  def get_highlights(self):
    '''return the important highlights that content have in a structured format'''
    if not self.text :
      return "[]"

    full_content = ''.join(self.text)

    prompt =f"""
    Analyze the following text:
        {full_content}
      Extract key highlights from the text.
      - Output: a JSON array of strings, where each string is a short bullet point highlight.
      - No explanations, no repetition
      - Focus on main ideas only
      Example: ["Highlight 1", "Highlight 2"]
    """
    response = self.__model.generate_content(prompt)
    return response.text

  def get_summary(self):
    '''return a concise summary of the content'''
    if not self.text :
      return f'Sorry, but there is nothing to summarize. Check file {self.file_path}'

    full_content = ''.join(self.text)

    prompt =f"""
    Summarize the following text concisely:
        {full_content}
    """
    response = self.__model.generate_content(prompt)
    return response.text

  def save_results(self, output_file_path='analysis_results.txt'):
    '''Saves the highlights and summary to a specified file.'''
    highlights = self.get_highlights()
    summary = self.get_summary()

    try:
      with open(output_file_path, 'w') as f:
        f.write("Highlights:\n")
        # Assuming highlights is a JSON string of a list of strings
        import json
        try:
            highlights_list = json.loads(highlights)
            for highlight in highlights_list:
                f.write(f"- {highlight}\n")
        except json.JSONDecodeError:
            f.write(f"Could not parse highlights as JSON: {highlights}\n")

        f.write("\nSummary:\n")
        f.write(summary)
      print(f"Analysis results successfully saved to '{output_file_path}'.")
    except Exception as e:
      print(f"An error occurred while saving results: {e}")

  def get_weights(self):
        '''Assigns a weight (1-10) to each line based on its importance.'''
        if not self.text:
            return {}

        # We send the lines with indices so the API can map them back correctly
        numbered_lines = "\n".join([f"{i}: {line.strip()}" for i, line in enumerate(self.text)])

        prompt = f"""
        Evaluate the following sentences. For each sentence index, assign a 'weight' from 1 to 10
        based on how much critical information it contains (10 being most important).

        Text:
        {numbered_lines}

        Output ONLY a valid JSON object where the key is the index and the value is the weight.
        Example: {{"0": 5, "1": 9, "2": 3}}
        """

        response = self.__model.generate_content(prompt)
        # Clean response text from markdown code blocks
        clean_json = response.text.replace('```json', '').replace('```', '').strip()

        try:
            return json.loads(clean_json)
        except:
            return {"error": "Failed to parse weights", "raw": response.text}

In [78]:
obj = TextAnalyser('/content/example.txt')

Successfully loaded file '/content/example.txt' (4156 bytes).


In [79]:
obj.get_highlights()

'[\n  "IIoT transformation of global manufacturing",\n  "Edge computing for reduced latency in real-time systems",\n  "Predictive maintenance to reduce downtime and costs",\n  "SQL-based pipelines for filtering large volumes of sensor data",\n  "Security challenges and the need for lightweight encryption",\n  "Integration of 5G accelerating IoT adoption",\n  "IoT applications in smart cities, agriculture, and healthcare",\n  "TinyML for deploying AI on low-power microcontrollers",\n  "Convergence of AI and IoT (AIoT) enabling self-healing systems",\n  "Ethical concerns regarding data privacy and standardization",\n  "Energy harvesting to extend device battery life",\n  "Specialized IoT core services from major cloud providers"\n]'

In [80]:
obj.get_summary()

'The Industrial Internet of Things (IIoT) and AI integration (AIoT) are transforming sectors like manufacturing, agriculture, and healthcare through real-time sensor data and edge computing. Key applications such as predictive maintenance and "self-healing" systems significantly reduce costs and optimize resources. However, the industry faces ongoing challenges in managing massive data volumes, ensuring cybersecurity, protecting user privacy, and achieving technical standardization across proprietary platforms.'

In [ ]:
obj.save_results()

Analysis results successfully saved to 'analysis_results.txt'.


In [ ]:
# # to test the size and confirmation of how os return file size
# import os

# path = '/content/python_return_guide.txt'

# size = os.path.getsize(path)

# size # returns in bytes confirmed

In [81]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create UI Elements
file_uploader = widgets.FileUpload(accept='.txt', multiple=False, description="Upload File")
analyze_btn = widgets.Button(description="Run Analysis", button_style='primary')
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()
        if not file_uploader.value:
            print("Please upload a file first.")
            return

        # Get uploaded file content
        uploaded_file = list(file_uploader.value.values())[0]
        file_content = uploaded_file['content'].decode('utf-8')

        # Temporary save to satisfy the class requirement
        temp_filename = "ui_upload.txt"
        with open(temp_filename, "w") as f:
            f.write(file_content)

        # Initialize and Run
        print("🤖 Gemini is analyzing...")
        analyser = TextAnalyser(temp_filename)

        highlights = analyser.get_highlights()
        summary = analyser.get_summary()
        weights = analyser.get_weights()

        print("\n" + "="*30)
        print("📊 SENTENCE WEIGHTS (Importance Score)")
        print("="*30)

        for i, line in enumerate(analyser.text):
            # Get weight for this line, default to 0 if not found
            weight = weights.get(str(i), 0)
            # Create a simple visual progress bar
            bar = "█" * int(weight)
            print(f"[{weight}/10] {bar.ljust(10)} | {line.strip()}")

        print("\n" + "="*30)
        print("📌 KEY HIGHLIGHTS")
        print("="*30)
        print(highlights)

        print("\n" + "="*30)
        print("📝 SUMMARY")
        print("="*30)
        print(summary)

        # Save results locally
        analyser.save_results("web_report.txt")

# 2. Set up the trigger
analyze_btn.on_click(on_button_clicked)

# 3. Display the UI
display(widgets.VBox([file_uploader, analyze_btn, output_area]))